<a href="https://colab.research.google.com/github/songwonni040122-ops/2026.08.Garbage-classification-model/blob/sub/2026_08_trash_classification_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. 데이터 불러오기

In [ ]:
import os
from google.colab import userdata

# 금고에서 토큰 받아오기
token = userdata.get('KAGGLE_KEY')

# 환경 변수에 토큰 등록
os.environ['KAGGLE_API_TOKEN'] = token

In [ ]:
import kagglehub

# garbage-classification 데이터셋 내려받기
path = kagglehub.dataset_download("asdasdasasdas/garbage-classification")

print("데이터 저장 경로:", path)

Using Colab cache for faster access to the 'garbage-classification' dataset.
데이터 저장 경로: /kaggle/input/garbage-classification


In [ ]:
# 데이터셋 폴더 구조확인
for root, dirs, files in os.walk(path):
    print(root, "| 폴더:", len(dirs), "| 파일:", len(files))

/kaggle/input/garbage-classification | 폴더: 2 | 파일: 5
/kaggle/input/garbage-classification/Garbage classification | 폴더: 1 | 파일: 0
/kaggle/input/garbage-classification/Garbage classification/Garbage classification | 폴더: 6 | 파일: 0
/kaggle/input/garbage-classification/Garbage classification/Garbage classification/metal | 폴더: 0 | 파일: 410
/kaggle/input/garbage-classification/Garbage classification/Garbage classification/glass | 폴더: 0 | 파일: 501
/kaggle/input/garbage-classification/Garbage classification/Garbage classification/paper | 폴더: 0 | 파일: 594
/kaggle/input/garbage-classification/Garbage classification/Garbage classification/trash | 폴더: 0 | 파일: 137
/kaggle/input/garbage-classification/Garbage classification/Garbage classification/cardboard | 폴더: 0 | 파일: 403
/kaggle/input/garbage-classification/Garbage classification/Garbage classification/plastic | 폴더: 0 | 파일: 482
/kaggle/input/garbage-classification/garbage classification | 폴더: 1 | 파일: 0
/kaggle/input/garbage-classification/garbage cla

In [ ]:
# 사용할 클래스 폴더 경로
data_dir = "/kaggle/input/garbage-classification/Garbage classification/Garbage classification"

# 클래스 6종 확인, 점검완료
print(os.listdir(data_dir))

['metal', 'glass', 'paper', 'trash', 'cardboard', 'plastic']


In [ ]:
import shutil
import os

# 원본 데이터 복사
src = data_dir

# Google Drive 내 저장할 경로 지정
dst = "/content/drive/MyDrive/MAIN/PROJECT/AI-Project/2026.08.Garbage-classification-model/Garbage-data"

# 폴더가 없을 시 새로 만들기 (환경이 바뀌면 동작하지 않을 경우를 대비)
os.makedirs(dst, exist_ok=True)

# 폴더 복사 (이미 있으면 덮어씀)
shutil.copytree(src, dst, dirs_exist_ok=True)

print("복사 완료")
print("드라이브에 저장된 클래스:", os.listdir(dst))

복사 완료
드라이브에 저장된 클래스: ['metal', 'glass', 'paper', 'trash', 'cardboard', 'plastic']


In [21]:
# Google Drive 폴더로 데이터 경로 설정
data_dir = "/content/drive/MyDrive/MAIN/PROJECT/AI-Project/2026.08.Garbage-classification-model/Garbage-data"

# 확인
print(os.listdir(data_dir))

['trash', 'cardboard', 'plastic', 'metal', 'glass', 'paper']


In [22]:
# ===== 기본환경 세팅 (데이터 가져오기/기본 도구 불러오기) =====

# 1. 기본 도구 불러오기
import os
import shutil

# 2. Google Drive 연결
from google.colab import drive
drive.mount('/content/drive')

# 3. 데이터 경로 지정
data_dir = "/content/drive/MyDrive/MAIN/PROJECT/AI-Project/2026.08.Garbage-classification-model/Garbage-data"

# 4. 세팅 확인
print("세팅 완료")
print("클래스:", os.listdir(data_dir))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
세팅 완료
클래스: ['trash', 'cardboard', 'plastic', 'metal', 'glass', 'paper']


In [23]:
from torchvision import datasets

# data_dir 폴더를 읽어서 데이터 묶음으로 만듦
# 하위 폴더(cardboard, glass 등) 이름을 클래스로 자동 인식
dataset = datasets.ImageFolder(data_dir)

# 인식된 클래스 이름 목록 출력
print("클래스:", dataset.classes)

# 데이터 묶음 내 이미지 개수 출력
print("이미지 총 개수:", len(dataset))

클래스: ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
이미지 총 개수: 2527


In [25]:
# 데이터 묶음에서 첫 번째(0번) 항목을 꺼냄
# 사진 한 장과 그 정답 번호가 같이 나옴 -> 각각 image, label에 담음
image, label = dataset[0]

# 꺼낸 정답 번호가 몇 번인지 출력
print("정답 번호:", label)

# 그 번호가 어떤 클래스 이름인지 출력 (번호를 이름으로 바꿔봄)
print("정답 이름:", dataset.classes[label])

정답 번호: 0
정답 이름: cardboard


In [26]:
from collections import Counter

# Counter로 각 번호(class 번호)가 몇 번씩 나오는지 셈
counts = Counter(dataset.targets)

# 번호 순서대로(0~5) 클래스 이름과 개수를 같이 출력
for label_num in range(len(dataset.classes)):
    class_name = dataset.classes[label_num]   # 번호 -> 이름
    count = counts[label_num]                 # 그 번호가 몇 장인지
    print(class_name, ":", count, "장")

cardboard : 403 장
glass : 501 장
metal : 410 장
paper : 594 장
plastic : 482 장
trash : 137 장


In [27]:
from torchvision import transforms

# 전처리 규칙 정의
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # 모든 사진을 가로 224, 세로 224로 크기 통일
    transforms.ToTensor(),  # 사진을 숫자 덩어리로 변환 (값 범위 0~1)
    transforms.Normalize(           # 값을 ResNet 기준에 맞게 재조정
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

# 위 규칙을 적용해서 dataset을 새 버전으로 덮어씀
dataset = datasets.ImageFolder(data_dir, transform=transform)

# 확인
image, label = dataset[0]
print("모양(shape):", image.shape)
print("값 최소:", image.min().item(), "/ 값 최대:", image.max().item())

모양(shape): torch.Size([3, 224, 224])
값 최소: -1.9466564655303955 / 값 최대: 2.129034996032715


In [28]:
from sklearn.model_selection import train_test_split

# 나눌 대상은 "사진 번호" 0부터 2526까지 (전체 인덱스)
indices = range(len(dataset))

# 각 사진의 정답 번호 목록 (층화 분할 기준)
labels = dataset.targets

# 1차 분할: 전체를 train(70%) 과 나머지(30%)로 나눔
# stratify=labels -> 클래스 비율을 유지하며 나누라는 뜻
# random_state=42 -> 나누는 방식 고정
train_idx, temp_idx = train_test_split(
    indices,
    test_size=0.3,
    stratify=labels,
    random_state=42,
)

# 2차 분할: 위에서 남긴 30%를 val 과 test로 절반씩 나눔 (각 15%)
# temp_idx에 해당하는 정답들만 골라 다시 층화 기준으로 사용
temp_labels = [labels[i] for i in temp_idx]
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.5,
    stratify=temp_labels,
    random_state=42,
)

# 각 묶음에 사진이 몇 장씩 들어갔는지 확인
print("학습용:", len(train_idx), "장")
print("검증용:", len(val_idx), "장")
print("테스트용:", len(test_idx), "장")

학습용: 1768 장
검증용: 379 장
테스트용: 380 장


In [29]:
from collections import Counter

# 각 묶음의 정답 번호만 골라냄
train_labels = [labels[i] for i in train_idx]
val_labels   = [labels[i] for i in val_idx]
test_labels  = [labels[i] for i in test_idx]

# 클래스별로 몇 장씩 들어갔는지 출력
print("클래스별 (학습 / 검증 / 테스트)")
for num in range(len(dataset.classes)):
    name = dataset.classes[num]
    tr = Counter(train_labels)[num]   # 학습용
    va = Counter(val_labels)[num]     # 검증용
    te = Counter(test_labels)[num]    # 테스트용
    print(f"{name:10s}: {tr:4d} / {va:3d} / {te:3d}")

클래스별 (학습 / 검증 / 테스트)
cardboard :  282 /  61 /  60
glass     :  350 /  75 /  76
metal     :  287 /  61 /  62
paper     :  416 /  89 /  89
plastic   :  337 /  72 /  73
trash     :   96 /  21 /  20


In [30]:
from torch.utils.data import Subset

#  두가지로 나누어 전처리 ---------------------------------------------

# 학습용 전처리 (나중에 필요시 데이터 증강 예정)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),  # 크기 통일
    transforms.ToTensor(),          # 숫자로 변환
    transforms.Normalize(           # 정규화
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

# 평가용 전처리 (검증·테스트용. 증강 x)
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

# 같은 폴더를 전처리만 다르게 두 번 Read -----------------------------

# 학습용 전처리가 붙은 전체 묶음
train_base = datasets.ImageFolder(data_dir, transform=train_transform)

# 평가용 전처리가 붙은 전체 묶음
eval_base  = datasets.ImageFolder(data_dir, transform=eval_transform)


# 번호표로 세 묶음 갈라내기 ------------------------------------------

# 학습용: 학습용 전처리 묶음에서 train 번호만 뽑음
train_dataset = Subset(train_base, train_idx)

# 검증용: 평가용 전처리 묶음에서 val 번호만 뽑음
val_dataset   = Subset(eval_base, val_idx)

# 테스트용: 평가용 전처리 묶음에서 test 번호만 뽑음
test_dataset  = Subset(eval_base, test_idx)

In [31]:
# 확인
print("학습용:", len(train_dataset), "장")
print("검증용:", len(val_dataset), "장")
print("테스트용:", len(test_dataset), "장")
print("클래스별 (학습 / 검증 / 테스트)")
for num in range(len(train_base.classes)):
    name = train_base.classes[num]
    tr = Counter(train_labels)[num]
    va = Counter(val_labels)[num]
    te = Counter(test_labels)[num]
    print(f"{name:10s}: {tr:4d} / {va:3d} / {te:3d}")

학습용: 1768 장
검증용: 379 장
테스트용: 380 장
클래스별 (학습 / 검증 / 테스트)
cardboard :  282 /  61 /  60
glass     :  350 /  75 /  76
metal     :  287 /  61 /  62
paper     :  416 /  89 /  89
plastic   :  337 /  72 /  73
trash     :   96 /  21 /  20


In [32]:
from torch.utils.data import DataLoader

# 배치 크기: 한 번에 32장씩 묶어서 처리 (하이퍼파라미터, 일단 무난한 값으로 시작)
batch_size = 32

# 학습용 로더: 학습 때는 순서를 섞음 (shuffle=True)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# 검증용 로더: 성적 측정용이라 섞을 필요 없음 (shuffle=False)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# 테스트용 로더: 마찬가지로 안 섞음
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 확인: 배치가 몇 개씩 나오는지, 배치 하나의 모양은 어떤지
print("학습용 배치 개수:", len(train_loader))
print("검증용 배치 개수:", len(val_loader))
print("테스트용 배치 개수:", len(test_loader))

# 학습용 로더에서 배치 하나만 꺼내서 모양 확인
images, labels_batch = next(iter(train_loader))
print("배치 이미지 모양:", images.shape)
print("배치 정답 모양:", labels_batch.shape)

학습용 배치 개수: 56
검증용 배치 개수: 12
테스트용 배치 개수: 12
배치 이미지 모양: torch.Size([32, 3, 224, 224])
배치 정답 모양: torch.Size([32])
